# r13 step D: verifier, suites, dev FINAL setting vs r8p, adopt or keep
About 2-3 h.

In [ ]:
import os, glob, json, hashlib, shutil, subprocess
W = '/kaggle/working'
PUB, PRIV = f'{W}/public', f'{W}/private'   # public: counts only, safe to commit; private: real question text, never commit
os.makedirs(PUB, exist_ok=True)
os.makedirs(PRIV, exist_ok=True)
REPO, BRANCH = '/kaggle/temp/Kinthread-llm-model', 'main'
R8P_SHA = '38c289d351d6517b5cde1bc9fb64ea97bd33eee193639d6351edda52fdfd7a52'


def sh(cmd):
    """Runs one shell command; stops the notebook if it fails."""
    print('$', cmd, flush=True)
    if subprocess.run(cmd, shell=True).returncode:
        raise SystemExit(f'STOP: command failed: {cmd}')


def stop(msg):
    raise SystemExit('STOP: ' + msg)


def par(*cmds):
    """Runs commands at once, one T4 each (one dev pass takes ~6 h; two in a row pass Kaggle's 12 h limit)."""
    procs = []
    for i, cmd in enumerate(cmds):
        print(f'$ [gpu {i % 2}]', cmd, flush=True)
        procs.append((cmd, subprocess.Popen(cmd, shell=True, env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(i % 2)})))
    bad = [cmd for cmd, p in procs if p.wait()]
    if bad:
        raise SystemExit(f'STOP: command failed: {bad[0]}')


def find(pattern):
    return sorted(glob.glob(f'/kaggle/input/**/{pattern}', recursive=True))


def one(pattern, what):
    got = find(pattern)
    if not got:
        stop(f'attach {what} (no {pattern} under /kaggle/input)')
    return got[0]


def sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


os.makedirs('/kaggle/temp', exist_ok=True)
if not os.path.exists(REPO):
    sh(f'git clone --depth 1 -b {BRANCH} https://github.com/ArAnirudh2901/Kinthread-llm-model.git {REPO}')
os.chdir(REPO)
tgz = find('real_questions_full.tgz')
if not tgz:
    stop('attach the private dataset kinthread-real-questions (it holds real_questions_full.tgz)')
sh(f'rm -rf data/real_questions && tar -xzf {tgz[0]}')
if not glob.glob('data/real_questions/*.jsonl'):
    stop('no real question files after unpacking real_questions_full.tgz')
print('repo and real questions ready')

R12 = one('runs/v10-r13/best.pt', 'the output of kinthread-r13-c-twins')   # r13 after twin training
A_DEV = one('private/r8p_dev_final.json', 'the output of kinthread-r9p-a-baseline')
B1 = json.load(open(one('public/B_r8p_summary.json', 'the output of kinthread-r9p-b1-suites-r8p')))
print('r13', sha(R12)[:12])

In [ ]:
par(f'python train_verifier.py --ckpt {R12} --n 1500 --out {PUB}/r13_verifier.json',
    f'python -m eval.gauntlet --ckpt {R12} --k 16 --out {PUB}/r13_gauntlet.json && '
    f'python -m eval.consistency --groups 300 --ckpt {R12} --k 16 --out {PUB}/r13_consistency.json && '
    f'python -m eval.evaluate --ckpt {R12} --sets unseen --out {PUB}/r13_unseen.json')
BASE = '--adaptive 1.0 --hybrid --cover --split_search --para 2 --k 16'
sh(f'python -m eval.run_real --ckpt {R12} --split dev {BASE} --verifier {PUB}/r13_verifier.json --out {PRIV}/r13_dev_final.json')
sh(f'python -m eval.compare_runs {A_DEV} {PRIV}/r13_dev_final.json --out {PUB}/compare_final.json')

In [ ]:
cmp = json.load(open(f'{PUB}/compare_final.json'))
g, c = (json.load(open(f'{PUB}/r13_{x}.json')) for x in ('gauntlet', 'consistency'))
suites_ok = (g['correct'] >= B1['gauntlet_correct'] - 0.01 and c['all variants agree'] >= B1['consistency_all_agree'] - 0.01)
decision = {'step': 'D', 'adopt': bool(cmp['adopt'] and suites_ok), 'config': 'final', 'extra_flags': '',
            'verifier': 'r13_verifier.json', 'dev': cmp,
            'gauntlet': {'r8p': B1['gauntlet_correct'], 'r13': g['correct']},
            'consistency_all_agree': {'r8p': B1['consistency_all_agree'], 'r13': c['all variants agree']},
            'suites_not_worse': suites_ok, 'r13_sha256': sha(R12),
            'rule': 'adopt = dev fair gain >= 2 points, wins > losses, McNemar p < 0.05, gauntlet and consistency not lower by more than 1 point'}
json.dump(decision, open(f'{PUB}/decision.json', 'w'), indent=1)
print(json.dumps(decision, indent=1))
print('D DONE: ADOPT r13' if decision['adopt'] else 'D DONE: KEEP r8p')